# TFG — DeepColony Pipeline
----
#### BSc Data Science and Engineering
#### Author: Mario Coronado Fernández

**Fixes (v2):**
- `SingleColonyCNN.forward()` now uses explicit `_conv_block()` method making the paper-correct order `Conv → LeakyReLU → LRN → MaxPool` auditable per layer, with a shape assertion.
- Plate-aware train/val split: colonies from the same plate never appear in both train and val sets, eliminating data leakage.
- Siamese impostor pair generation now uses species-count-weighted sampling so rare species (e.g. *Serratia marcescens*, 41 colonies) appear proportionally alongside common ones (e.g. *E. coli*, 4240 colonies).
- **Checkpoint added to SingleColonyCNN**. Training resumes from the correct epoch and LR after any Colab disconnection.
- Training curves and confusion matrix generated from saved history/model without retraining.

### Necessary Imports

In [1]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import pickle
import random
import itertools
import numpy as np
from sklearn.cluster import MeanShift, estimate_bandwidth
from sklearn.metrics import v_measure_score, confusion_matrix, ConfusionMatrixDisplay
from collections import defaultdict

# Reproducibility
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    for i in range(n):
        print(f'GPU {i}           : {torch.cuda.get_device_name(i)}')
    print(f'Total GPUs      : {n}')

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU 0           : NVIDIA GeForce RTX 4050 Laptop GPU
Total GPUs      : 1


#### Connect to Colab & mount Drive

In [6]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    ZIP_PATH   = '/content/drive/MyDrive/TFG/data_tfg.zip'
    WORK_DIR   = '/content/data_lab'
    OUTPUT_DIR = '/content/drive/MyDrive/TFG/resultados_entrenamiento'
    os.makedirs(WORK_DIR,   exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if not os.path.exists(os.path.join(WORK_DIR, 'metadata.json')):
        print('Descomprimiendo dataset...')
        os.system(f'unzip -q "{ZIP_PATH}" -d "{WORK_DIR}"')
        print('Completado!')
    else:
        print('Los datos ya están listos.')
else:
    # ── LOCAL (VSCode) ─────────────────────────────────────────
    WORK_DIR   = r'C:\Users\User\Desktop\TFG\Data'          # ← adjust
    OUTPUT_DIR = r'C:\Users\User\Desktop\TFG\results'        # ← adjust

    os.makedirs(WORK_DIR,   exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if not os.path.exists(os.path.join(WORK_DIR, 'metadata.json')):
        print('Descomprimiendo dataset...')
        import zipfile
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(WORK_DIR)
        print('Completado!')
    else:
        print('Los datos ya están listos.')

print(f'WORK_DIR   = {WORK_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

Los datos ya están listos.
WORK_DIR   = C:\Users\User\Desktop\TFG\Data
OUTPUT_DIR = C:\Users\User\Desktop\TFG\results


---
## Level 2 — Single Colony CNN
### Architecture


In [7]:
# ==================== SINGLE COLONY CNN ====================
# Paper: Supp. Note 1 — four conv layers, Leaky ReLU, LRN, MaxPool(stride=2)
# Operation order per paper: Conv → LeakyReLU → LRN → MaxPool

class SingleColonyCNN(nn.Module):
    def __init__(self):
        super(SingleColonyCNN, self).__init__()
        # Conv layers (filters match paper exactly)
        self.conv1 = nn.Conv2d(3,   20,  kernel_size=5)  # 128 → 124
        self.conv2 = nn.Conv2d(20,  50,  kernel_size=5)  # → 58 (after pool)
        self.conv3 = nn.Conv2d(50,  100, kernel_size=4)  # → 26 (after pool)
        self.conv4 = nn.Conv2d(100, 200, kernel_size=4)  # → 10 (after pool)

        # LRN and MaxPool are shared across all conv blocks
        # Paper: 'outputs of Leaky ReLU activations are normalized using LRN'
        # Paper: 'non-overlapping max-pooling layer with stride equal to 2'
        self.lrn     = nn.LocalResponseNorm(size=5)
        self.pool    = nn.MaxPool2d(kernel_size=2, stride=2)  # non-overlapping

        # FC layers
        # After 4x (conv + pool): 128→62→29→13→5, so 200*5*5 = 5000
        self.dropout = nn.Dropout(p=0.75)   # paper: dropout rate 0.75
        self.fc1     = nn.Linear(200 * 5 * 5, 500)
        self.fc2     = nn.Linear(500, 32)   # 32 species

    def _conv_block(self, x, conv_layer):
        """Single conv block in paper-specified order:
           Conv → LeakyReLU (slope=0.01) → LRN → MaxPool(stride=2)
        """
        x = conv_layer(x)
        x = F.leaky_relu(x, negative_slope=0.01)
        x = self.lrn(x)
        x = self.pool(x)
        return x

    def forward(self, x, _check_shape=False):
        x = self._conv_block(x, self.conv1)   # (B,3,128,128)  → (B,20,62,62)
        x = self._conv_block(x, self.conv2)   # (B,20,62,62)   → (B,50,29,29)
        x = self._conv_block(x, self.conv3)   # (B,50,29,29)   → (B,100,13,13)
        x = self._conv_block(x, self.conv4)   # (B,100,13,13)  → (B,200,5,5)

        # One-time shape check — run before first training step
        if _check_shape:
            assert x.shape[1:] == (200, 5, 5), (
                f'Unexpected feature map shape: {x.shape}. '
                f'Expected (B, 200, 5, 5). Check input size and conv config.')
            print(f'  Shape check passed: feature maps are {tuple(x.shape[1:])}')

        x = x.view(x.size(0), -1)             # flatten → (B, 5000)
        x = self.dropout(x)
        x = F.leaky_relu(self.fc1(x), negative_slope=0.01)
        return self.fc2(x)                     # raw logits; softmax applied at loss/inference

    def get_pIDv(self, x):
        """Returns softmax probability vector (pIDv) — use at inference time only."""
        self.eval()
        with torch.no_grad():
            return torch.softmax(self.forward(x), dim=1)


# ==================== ADAPTIVE PADDING ====================
# Pads colony crops to square + margin BEFORE resizing.
# This preserves absolute colony size (a discriminative feature per paper).
class AdaptivePadding:
    def __init__(self, margin=20, fill=0):
        self.margin = margin
        self.fill   = fill

    def __call__(self, img):
        w, h     = img.size
        max_side = max(w, h)
        pad_w    = max_side - w + self.margin
        pad_h    = max_side - h + self.margin
        padding  = (pad_w // 2, pad_h // 2,
                    pad_w - pad_w // 2, pad_h - pad_h // 2)
        return transforms.functional.pad(img, padding, fill=self.fill)


# Check on architecture dimensions
with torch.no_grad():
    _dummy = torch.randn(2, 3, 128, 128)
    _m     = SingleColonyCNN()
    _out   = _m(_dummy, _check_shape=True)
    print(f'  Output logits shape: {_out.shape}  (expected (2, 32))')
    del _dummy, _m, _out
print('SingleColonyCNN architecture OK.')

  Shape check passed: feature maps are (200, 5, 5)
  Output logits shape: torch.Size([2, 32])  (expected (2, 32))
SingleColonyCNN architecture OK.


### Dataset

In [8]:
# ==================== COLONY DATASET ====================
class ColonyDataset(Dataset):
    def __init__(self, metadata_path, dataset_path, transform=None,
                 species_to_idx=None):
        with open(metadata_path, 'r') as f:
            data = json.load(f)
        self.metadata     = list(data['patch_list'].values())
        self.dataset_path = dataset_path
        self.transform    = transform

        # Build mapping from the FULL dataset so every subset shares
        # identical label indices — critical for multi-run consistency.
        if species_to_idx is None:
            unique_species      = sorted(set(item['species'] for item in self.metadata))
            self.species_to_idx = {sp: i for i, sp in enumerate(unique_species)}
        else:
            self.species_to_idx = species_to_idx

        self.idx_to_species = {i: sp for sp, i in self.species_to_idx.items()}

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        item = self.metadata[idx]
        try:
            image = Image.open(
                os.path.join(self.dataset_path, item['filename'])
            ).convert('RGB')
        except Exception:
            image = Image.new('RGB', (128, 128))
        if self.transform:
            image = self.transform(image)
        return image, self.species_to_idx[item['species']]

### Plate-Aware Train / Val Split

The paper splits at the **plate level**, not the colony level.
Splitting randomly by colony index causes data leakage: the model may see colonies from the
same plate in both train and val, inflating validation accuracy.

This function groups colony indices by `plate_n`, shuffles **plates**, then assigns whole
plates to train or val — guaranteeing zero overlap at the colony level.

In [9]:
def make_plate_aware_split(full_dataset, val_ratio=0.2, test_ratio=0.2, seed=42):
    """
    3-way plate-aware split: train / val / test.
    Paper: '60/20/20 subdivision performed at the plate level.'

    Returns:
        tr_idx, va_idx, te_idx : lists of integer indices into the dataset
        train_plates, val_plates, test_plates : sets of plate_n strings
    """
    rng = random.Random(seed)

    plate_to_indices = defaultdict(list)
    for i, item in enumerate(full_dataset.metadata):
        plate_to_indices[item['plate_n']].append(i)

    plates = list(plate_to_indices.keys())
    rng.shuffle(plates)

    n_test  = int(len(plates) * test_ratio)
    n_val   = int(len(plates) * val_ratio)

    test_pls  = set(plates[:n_test])
    val_pls   = set(plates[n_test : n_test + n_val])
    train_pls = set(plates[n_test + n_val:])

    tr_idx = [i for p in train_pls for i in plate_to_indices[p]]
    va_idx = [i for p in val_pls   for i in plate_to_indices[p]]
    te_idx = [i for p in test_pls  for i in plate_to_indices[p]]

    # Sanity: zero overlap
    assert len(set(tr_idx) & set(va_idx)) == 0, 'BUG: train/val overlap!'
    assert len(set(tr_idx) & set(te_idx)) == 0, 'BUG: train/test overlap!'
    assert len(set(va_idx) & set(te_idx)) == 0, 'BUG: val/test overlap!'

    print(f'Plate-aware 3-way split:')
    print(f'  Total plates : {len(plates)}')
    print(f'  Train plates : {len(train_pls):4d}  →  {len(tr_idx):6d} colonies  (~{len(tr_idx)/sum([len(tr_idx),len(va_idx),len(te_idx)])*100:.0f}%)')
    print(f'  Val   plates : {len(val_pls):4d}  →  {len(va_idx):6d} colonies  (~{len(va_idx)/sum([len(tr_idx),len(va_idx),len(te_idx)])*100:.0f}%)')
    print(f'  Test  plates : {len(test_pls):4d}  →  {len(te_idx):6d} colonies  (~{len(te_idx)/sum([len(tr_idx),len(va_idx),len(te_idx)])*100:.0f}%)')
    print(f'  Leakage checks: PASSED')

    return tr_idx, va_idx, te_idx, train_pls, val_pls, test_pls

### LR Scheduler, Weight Init & Training Loop

In [10]:
# ==================== LR SCHEDULER ====================
# Replicates the paper's exact schedule:
#   - Exponential decay: lr *= 0.9999 per iteration
#   - One-time halving at 50,000 iterations
# Custom scheduler is necessary because PyTorch's ExponentialLR operates
# per-epoch, not per-iteration.
class PaperLRScheduler:
    def __init__(self, optimizer, initial_lr=0.01,
                 decay_rate=0.9999, halve_at=50000):
        self.optimizer  = optimizer
        self.initial_lr = initial_lr
        self.decay_rate = decay_rate
        self.halve_at   = halve_at
        self.iteration  = 0
        self.halved     = False
        for pg in optimizer.param_groups:
            pg['lr'] = initial_lr

    def step(self):
        self.iteration += 1
        lr = self.initial_lr * (self.decay_rate ** self.iteration)
        if self.iteration >= self.halve_at and not self.halved:
            lr /= 2.0
            self.halved = True
            print(f'\n[Iter {self.iteration}] LR halved → {lr:.6f}')
        for pg in self.optimizer.param_groups:
            pg['lr'] = lr

    def get_last_lr(self):
        return [pg['lr'] for pg in self.optimizer.param_groups]


# ==================== WEIGHT INIT ====================
def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)


# ==================== TRAIN SINGLE COLONY CNN ====================
def train_single_colony_cnn(model, train_loader, val_loader,
                             num_epochs=70, device='cuda',
                             save_path='best_model.pth',
                             checkpoint_path='single_cnn_checkpoint.pth'):

    model     = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01,
                          momentum=0.9, weight_decay=0.0005)
    scheduler = PaperLRScheduler(optimizer)

    # ── Resumption block ─────────────────────────────────────────────
    start_epoch  = 0
    best_val_acc = 0.0
    history      = {'train_losses': [], 'val_losses': [],
                    'train_accs':   [], 'val_accs':   []}

    if os.path.exists(checkpoint_path):
        print(f'[Checkpoint found] Resuming: {checkpoint_path}')
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        # Restore scheduler state — without this, LR resets to 0.01
        scheduler.iteration = ckpt['scheduler_iteration']
        scheduler.halved    = ckpt['scheduler_halved']
        start_epoch         = ckpt['epoch'] + 1
        best_val_acc        = ckpt['best_val_acc']
        history             = ckpt.get('history', history)
        print(f'  Resumed at epoch {start_epoch} | '
              f'Best val acc so far: {best_val_acc:.2f}% | '
              f'Scheduler iter: {scheduler.iteration}')
    else:
        print('No checkpoint found — starting from scratch.')
    # ─────────────────────────────────────────────────────────────────

    for epoch in range(start_epoch, num_epochs):  # ← start_epoch, not 0

        # ── Train ────────────────────────────────────────────────────
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        for imgs, labels in tqdm(train_loader,
                                 desc=f'Epoch {epoch+1}/{num_epochs} [Train]'):
            imgs, labels = imgs.to(device), labels.to(device)
            # Forward pass
            out  = model(imgs)
            loss = criterion(out, labels)
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()          # per-iteration as the paper specifies
            run_loss += loss.item()
            correct  += (out.argmax(1) == labels).sum().item()
            total    += labels.size(0)

        train_loss = run_loss / len(train_loader)
        train_acc  = 100.0 * correct / total
        history['train_losses'].append(train_loss)
        history['train_accs'].append(train_acc)

        # ── Validate ─────────────────────────────────────────────────
        model.eval()
        val_loss, vc, vt = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                out       = model(imgs)
                loss      = criterion(out, labels)
                val_loss += loss.item()
                vc       += (out.argmax(1) == labels).sum().item()
                vt       += labels.size(0)

        val_acc  = 100.0 * vc / vt
        val_loss = val_loss / len(val_loader)
        history['val_losses'].append(val_loss)
        history['val_accs'].append(val_acc)

        current_lr = scheduler.get_last_lr()[0]
        print(f'  Epoch {epoch+1:3d}/{num_epochs} | '
              f'Train Loss: {train_loss:.4f}  Acc: {train_acc:.2f}% | '
              f'Val Loss: {val_loss:.4f}  Acc: {val_acc:.2f}%  '
              f'(best: {best_val_acc:.2f}%)  LR: {current_lr:.6f}')

        # ── Save checkpoint every epoch ──────────────────
        torch.save({
            'epoch'                : epoch,
            'model_state_dict'     : model.state_dict(),
            'optimizer_state_dict' : optimizer.state_dict(),
            'scheduler_iteration'  : scheduler.iteration,
            'scheduler_halved'     : scheduler.halved,
            'best_val_acc'         : best_val_acc,
            'history'              : history,
            'species_to_idx'       : train_loader.dataset.species_to_idx,
            'idx_to_species'       : train_loader.dataset.idx_to_species,
        }, checkpoint_path)

        # ── Save best model ──────────
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch'           : epoch,
                'model_state_dict': model.state_dict(),
                'val_acc'         : val_acc,
                'species_to_idx'  : train_loader.dataset.species_to_idx,
                'idx_to_species'  : train_loader.dataset.idx_to_species,
            }, save_path)
            print(f'  ✓ Best model saved  (val_acc={val_acc:.2f}%)')

    return history

### Main — Train Single Colony CNN

In [ ]:
# ==================== MAIN — SINGLE COLONY CNN ====================
METADATA_PATH    = os.path.join(WORK_DIR,   'metadata.json')
DATASET_PATH     = os.path.join(WORK_DIR,   'Dataset')
MODEL_SAVE_PATH  = os.path.join(OUTPUT_DIR, 'best_single_colony_model.pth')
CHECKPOINT_PATH  = os.path.join(OUTPUT_DIR, 'single_cnn_checkpoint.pth')

BATCH_SIZE = 64
NUM_EPOCHS = 200
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

# Transforms
t_train = transforms.Compose([
    AdaptivePadding(20),
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
])
t_val = transforms.Compose([
    AdaptivePadding(20),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

# Load full dataset (no transform — only used to inspect metadata for split)
full_ds = ColonyDataset(METADATA_PATH, DATASET_PATH, transform=None)

# ── Plate-aware split ──────────────────────────────────
tr_idx, va_idx, te_idx, TRAIN_PLATES, VAL_PLATES, TEST_PLATES = make_plate_aware_split(
    full_ds, val_ratio=0.2, test_ratio=0.2, seed=GLOBAL_SEED
)

# Build train/val/test datasets sharing the FULL mapping
tr_ds = ColonyDataset(METADATA_PATH, DATASET_PATH,
                      transform=t_train,
                      species_to_idx=full_ds.species_to_idx)
va_ds = ColonyDataset(METADATA_PATH, DATASET_PATH,
                      transform=t_val,
                      species_to_idx=full_ds.species_to_idx)
te_ds = ColonyDataset(METADATA_PATH, DATASET_PATH,
                      transform=t_val,          # ← NO augmentation on test
                      species_to_idx=full_ds.species_to_idx)

tr_ds.metadata = [full_ds.metadata[i] for i in tr_idx]
va_ds.metadata = [full_ds.metadata[i] for i in va_idx]
te_ds.metadata = [full_ds.metadata[i] for i in te_idx]

tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=2, pin_memory=True)
va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=2, pin_memory=True)
te_loader = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=2, pin_memory=True)

# Build model — only apply Xavier init if starting from scratch
model = SingleColonyCNN()
if not os.path.exists(CHECKPOINT_PATH):
    model.apply(init_weights)
    print('Xavier initialisation applied.')
else:
    print('Checkpoint exists — skipping Xavier init (weights restored from checkpoint).')

# Train (resumes automatically if checkpoint exists)
single_cnn_history = train_single_colony_cnn(
    model, tr_loader, va_loader,
    num_epochs=NUM_EPOCHS,
    device=DEVICE,
    save_path=MODEL_SAVE_PATH,
    checkpoint_path=CHECKPOINT_PATH,
)

# Save history separately so plots can be regenerated without retraining
with open(os.path.join(OUTPUT_DIR, 'single_cnn_history.pkl'), 'wb') as f:
    pickle.dump(single_cnn_history, f)
print('Training complete. History saved.')

Plate-aware 3-way split:
  Total plates : 1351
  Train plates :  811  →   15747 colonies  (~60%)
  Val   plates :  270  →    4864 colonies  (~19%)
  Test  plates :  270  →    5602 colonies  (~21%)
  Leakage checks: PASSED
Xavier initialisation applied.
No checkpoint found — starting from scratch.


Epoch 1/200 [Train]:   0%|          | 0/247 [00:00<?, ?it/s]

### Training Curves — Single Colony CNN

In [ ]:
# ==================== PLOT TRAINING CURVES ====================
def plot_training_curves(history, title='Single Colony CNN', save_dir=None):
    """Plot loss and accuracy curves from history dict.
    Can be called with history loaded from disk — no retraining needed.
    """
    epochs = range(1, len(history['train_losses']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    # Loss
    axes[0].plot(epochs, history['train_losses'], label='Train', color='steelblue')
    axes[0].plot(epochs, history['val_losses'],   label='Val',   color='darkorange')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # Accuracy
    axes[1].plot(epochs, history['train_accs'], label='Train', color='steelblue')
    axes[1].plot(epochs, history['val_accs'],   label='Val',   color='darkorange')
    best_epoch = int(np.argmax(history['val_accs'])) + 1
    best_acc   = max(history['val_accs'])
    axes[1].axvline(best_epoch, color='red', linestyle='--', alpha=0.6,
                    label=f'Best val epoch {best_epoch} ({best_acc:.1f}%)')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    if save_dir:
        path = os.path.join(save_dir, 'single_cnn_curves.png')
        plt.savefig(path, dpi=150, bbox_inches='tight')
        print(f'Saved: {path}')
    plt.show()


# Load history from disk (works after any disconnection)
history_path = os.path.join(OUTPUT_DIR, 'single_cnn_history.pkl')
if os.path.exists(history_path):
    with open(history_path, 'rb') as f:
        _h = pickle.load(f)
    plot_training_curves(_h, title='Single Colony CNN — Training Curves',
                         save_dir=OUTPUT_DIR)
else:
    print('History file not found — run training first.')

NameError: name 'os' is not defined

### Confusion Matrix — Single Colony CNN

Loads the best saved model from Drive and evaluates it on the validation set.
No retraining required.

In [ ]:
# ==================== CONFUSION MATRIX (no retraining) ====================
def evaluate_and_plot_confusion(model_path, val_loader, device,
                                 save_dir=None, top_n=None):
    """
    Load best model from disk and compute confusion matrix on val_loader.
    top_n: if set, only display the top_n most frequent classes (cleaner plot).
    """
    ckpt = torch.load(model_path, map_location=device)
    idx_to_species = ckpt['idx_to_species']

    net = SingleColonyCNN().to(device)
    net.load_state_dict(ckpt['model_state_dict'])
    net.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc='Evaluating'):
            imgs   = imgs.to(device)
            preds  = net(imgs).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    overall_acc = 100.0 * (all_preds == all_labels).mean()
    print(f'Overall val accuracy: {overall_acc:.2f}%')

    # Short species labels for readability
    n_classes   = len(idx_to_species)
    short_names = []
    for i in range(n_classes):
        parts = idx_to_species[i].split()
        short_names.append(f'{parts[0][0]}. {" ".join(parts[1:])}')

    cm = confusion_matrix(all_labels, all_preds, labels=list(range(n_classes)))

    fig, ax = plt.subplots(figsize=(18, 16))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=short_names)
    disp.plot(ax=ax, colorbar=True, xticks_rotation=90, values_format='d')
    ax.set_title(f'Confusion Matrix — Single Colony CNN\n'
                 f'Val Accuracy: {overall_acc:.2f}%', fontsize=13)
    plt.tight_layout()

    if save_dir:
        path = os.path.join(save_dir, 'single_cnn_confusion_matrix.png')
        plt.savefig(path, dpi=150, bbox_inches='tight')
        print(f'Saved: {path}')
    plt.show()
    return cm, overall_acc


# ── Confusion matrix on the TEST set ─────────────────────────
# This mirrors the paper: model was selected on val, never saw test during training.
if os.path.exists(MODEL_SAVE_PATH):
    cm, acc = evaluate_and_plot_confusion(
        MODEL_SAVE_PATH, te_loader, DEVICE, save_dir=OUTPUT_DIR   # ← te_loader, not va_loader
    )
else:
    print('Best model not found — run training first.')

NameError: name 'MODEL_SAVE_PATH' is not defined

---
## Level 3 — Siamese CNN (Architecture + Training)

The Siamese CNN already had checkpoint support. The fix here is in pair generation (Issue 5).

In [ ]:
# ==================== SIAMESE CNN ====================
# KEY ARCHITECTURE NOTES FROM PAPER:
#   - MaxPool stride=1 (not 2) → feature maps stay large intentionally
#     ('the size of the convolutional feature maps before FC are quite large')
#   - PReLU on top of EVERY layer including the output layer
#   - embedding_dim=15 as stated in the Methods section

class SiameseCNN(nn.Module):
    def __init__(self, embedding_dim=15):
        super(SiameseCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 20, kernel_size=5)   # 128→124
        self.conv2 = nn.Conv2d(20, 50, kernel_size=5)  # 123→119

        # PReLU on every layer (including output)
        self.prelu_c1  = nn.PReLU()
        self.prelu_c2  = nn.PReLU()
        self.prelu_fc1 = nn.PReLU()
        self.prelu_fc2 = nn.PReLU()

        # stride=1 intentionally keeps feature maps large
        # (too much distortion invariance is detrimental for similarity)
        # 128 → conv1 → 124 → pool(k=2,s=1) → 123 → conv2 → 119
        # flatten: 50 * 119 * 119 = 708,050
        self.pool    = nn.MaxPool2d(kernel_size=2, stride=1)
        self.fc1     = nn.Linear(50 * 119 * 119, 500)
        self.fc2     = nn.Linear(500, embedding_dim)
        self.dropout = nn.Dropout(0.5)

    def forward_one(self, x):
        x = self.pool(self.prelu_c1(self.conv1(x)))    # [B, 20, 123, 123]
        x = self.prelu_c2(self.conv2(x))               # [B, 50, 119, 119]
        x = x.view(x.size(0), -1)                      # [B, 708050]
        x = self.dropout(self.prelu_fc1(self.fc1(x)))  # [B, 500]
        x = self.prelu_fc2(self.fc2(x))                # [B, 15]
        return x

    def forward(self, x1, x2):
        return self.forward_one(x1), self.forward_one(x2)


# ==================== CONTRASTIVE LOSS ====================
# Eq. 1 from Supp. Note 2:
# E = (1/2N) Σ [ y*||x-xp||² + (1-y)*max(1-||x-xp||², 0)² ]
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, out1, out2, label):
        # Euclidean distance (not squared)
        d   = torch.sqrt(torch.sum((out1 - out2) ** 2, dim=1) + 1e-8)

        # Chopra et al. original formula:
        # positive pairs: penalise large d
        # negative pairs: penalise d < margin
        pos = label * d ** 2
        neg = (1 - label) * torch.clamp(self.margin - d, min=0.0) ** 2
        return torch.mean(pos + neg) / 2.0

### Pair Generation & Dataset

In [ ]:
# ==================== SIAMESE DATASET ====================
class SiameseDataset(Dataset):
    def __init__(self, pairs, labels, root_dir, transform=None):
        self.pairs     = pairs
        self.labels    = labels
        self.root_dir  = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        f1, f2 = self.pairs[idx]
        label  = self.labels[idx]
        img1   = Image.open(os.path.join(self.root_dir, f1)).convert('RGB')
        img2   = Image.open(os.path.join(self.root_dir, f2)).convert('RGB')
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)
        return img1, img2, torch.tensor(label, dtype=torch.float32)


# ==================== PAIR GENERATION ====================
# OLD problem: random.sample(sp_names, 2) gave every species equal probability.
# E. coli (4240 colonies) and Serratia marcescens (41 colonies) were treated
# identically, causing rare species to be underrepresented in impostor pairs.
#
# FIX: random.choices(..., weights=sp_counts) weights each species by its
# colony count, so the impostor distribution mirrors the real data distribution.
def generate_pairs_from_json(json_path, max_pairs_per_plate=50,
                              restrict_to_plates=None):
    """
    Pair generation rules (paper):
      - Genuine  (label=1): same plate AND same strain
      - Impostor (label=0): different species, weighted by colony count
      - Same-strain/different-plate: EXCLUDED from genuine pairs
      - 50/50 balance enforced

    Args:
        restrict_to_plates : optional set of plate_n strings to restrict sampling
                             (used to generate train/val pairs from pre-split plates)
    """
    print(f'Generating pairs (max {max_pairs_per_plate} genuine per plate)...')
    with open(json_path, 'r') as f:
        data = json.load(f)
    samples = list(data['patch_list'].values())

    # Optionally restrict to a plate subset
    if restrict_to_plates is not None:
        samples = [s for s in samples if s['plate_n'] in restrict_to_plates]
        print(f'  Restricted to {len(restrict_to_plates)} plates → {len(samples)} colonies')

    plates         = defaultdict(list)
    species_groups = defaultdict(list)
    for s in samples:
        plates[s['plate_n']].append(s)
        species_groups[s['species']].append(s)

    # ── Genuine pairs: same plate AND same species ────────────────────
    genuine_pairs = []
    for p_name, p_samples in plates.items():
        possible = [
            (p1['filename'], p2['filename'])
            for p1, p2 in itertools.combinations(p_samples, 2)
            if p1['species'] == p2['species']
        ]
        if len(possible) > max_pairs_per_plate:
            possible = random.sample(possible, max_pairs_per_plate)
        genuine_pairs.extend(possible)
    print(f'  Genuine pairs  : {len(genuine_pairs)}')

    # ── Impostor pairs: different species, weighted by colony count ───
    # FIX: weight by species size so rare species appear proportionally
    sp_names  = list(species_groups.keys())
    sp_counts = [len(species_groups[sp]) for sp in sp_names]

    impostor_pairs = []
    target = len(genuine_pairs)
    while len(impostor_pairs) < target:
        # random.choices allows weighted sampling (with replacement)
        sp1, sp2 = random.choices(sp_names, weights=sp_counts, k=2)
        if sp1 == sp2:        # reject same-species draws
            continue
        img1 = random.choice(species_groups[sp1])['filename']
        img2 = random.choice(species_groups[sp2])['filename']
        impostor_pairs.append((img1, img2))

    print(f'  Impostor pairs : {len(impostor_pairs)}')

    # ── Combine and shuffle ───────────────────────────────────────────
    all_pairs  = genuine_pairs + impostor_pairs
    all_labels = [1.0] * len(genuine_pairs) + [0.0] * len(impostor_pairs)
    combined   = list(zip(all_pairs, all_labels))
    random.shuffle(combined)
    pairs, labels = zip(*combined)
    print(f'  Total          : {len(pairs)} pairs  (50/50 balance)')
    return list(pairs), list(labels)

### Siamese Training Loop

In [ ]:
# ==================== TRAIN SIAMESE ====================
# Checkpoint support was already present — no changes to the loop.
# The fix is upstream in generate_pairs_from_json (weighted impostor sampling).
def train_siamese_network(model, train_loader, val_loader, num_epochs,
                           device, save_path, checkpoint_path):
    # ── DEBUG (remove after fix confirmed) ──
    print(f'  [DEBUG] checkpoint_path = {checkpoint_path!r}')
    print(f'  [DEBUG] file exists     = {os.path.exists(checkpoint_path)}')
    # ────────────────────────────────────────

    model     = model.to(device)
    criterion = ContrastiveLoss(margin=1.0)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0005)
    start_epoch, best_val_loss = 0, float('inf')
    siamese_history = {'train_losses': [], 'val_losses': [],
                       'train_accs':   [], 'val_accs':   []}

    if os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        start_epoch     = ckpt['epoch'] + 1
        best_val_loss   = ckpt.get('best_val_loss', float('inf'))
        siamese_history = ckpt.get('history', siamese_history)
        print(f'Resuming from epoch {start_epoch}')
    else:
        print('Starting from scratch.')

    for epoch in range(start_epoch, num_epochs):
        print(f'  [best so far: {best_val_loss:.4f}]')
        # Train
        model.train()
        t_loss, correct, total = 0.0, 0, 0
        for img1, img2, label in tqdm(train_loader,
                                      desc=f'Epoch {epoch+1}/{num_epochs}'):
            img1, img2, label = img1.to(device), img2.to(device), label.to(device)
            optimizer.zero_grad()
            o1, o2 = model(img1, img2)
            loss   = criterion(o1, o2, label)
            loss.backward()
            optimizer.step()
            t_loss  += loss.item()
            d        = torch.sqrt(torch.sum((o1 - o2) ** 2, dim=1) + 1e-8)
            correct += ((d < 0.5).float() == label).sum().item()
            total   += label.size(0)

        # Val
        model.eval()
        v_loss, vc, vt = 0.0, 0, 0
        with torch.no_grad():
            for img1, img2, label in val_loader:
                img1, img2, label = img1.to(device), img2.to(device), label.to(device)
                o1, o2 = model(img1, img2)
                loss   = criterion(o1, o2, label)
                v_loss += loss.item()
                d2      = torch.sum((o1 - o2) ** 2, dim=1)
                vc     += ((d2 < 0.5).float() == label).sum().item()
                vt     += label.size(0)

        avg_tl = t_loss / len(train_loader)
        avg_vl = v_loss / len(val_loader)
        t_acc  = 100.0 * correct / total
        v_acc  = 100.0 * vc / vt

        siamese_history['train_losses'].append(avg_tl)
        siamese_history['val_losses'].append(avg_vl)
        siamese_history['train_accs'].append(t_acc)
        siamese_history['val_accs'].append(v_acc)

        print(f'Epoch {epoch+1}: '
              f'Train Loss={avg_tl:.4f} Acc={t_acc:.1f}% | '
              f'Val Loss={avg_vl:.4f} Acc={v_acc:.1f}%')

        if avg_vl < best_val_loss:
            best_val_loss = avg_vl
            torch.save(model.state_dict(), save_path)
            print(f'  → Best model saved (Val Loss: {avg_vl:.4f})')
        # Checkpoint every epoch
        torch.save({
            'epoch'               : epoch,
            'model_state_dict'    : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss'       : best_val_loss,
            'history'             : siamese_history,
        }, checkpoint_path)

    return siamese_history

### Main — Train Siamese CNN

In [ ]:
# ==================== RESTORE PLATE SPLIT (if lost after disconnection) ====================
# TRAIN_PLATES / VAL_PLATES / TEST_PLATES are defined in the Single Colony CNN main cell.
# If that cell hasn't been run yet this session, recompute the split here.
# Uses the same seed → identical plate assignments every time.

import os, json
from collections import defaultdict

if 'TRAIN_PLATES' not in dir() or 'VAL_PLATES' not in dir() or 'TEST_PLATES' not in dir():
    print('Plate split variables not found — recomputing from metadata...')

    METADATA_PATH = os.path.join(WORK_DIR, 'metadata.json')
    DATASET_PATH  = os.path.join(WORK_DIR, 'Dataset')

    full_ds = ColonyDataset(METADATA_PATH, DATASET_PATH, transform=None)

    tr_idx, va_idx, te_idx, TRAIN_PLATES, VAL_PLATES, TEST_PLATES = make_plate_aware_split(
        full_ds, val_ratio=0.2, test_ratio=0.2, seed=GLOBAL_SEED
    )
    print('Split restored.')
else:
    print('Plate split already in memory — skipping recompute.')

print(f'  TRAIN_PLATES : {len(TRAIN_PLATES)}')
print(f'  VAL_PLATES   : {len(VAL_PLATES)}')
print(f'  TEST_PLATES  : {len(TEST_PLATES)}')

Plate split variables not found — recomputing from metadata...
Plate-aware 3-way split:
  Total plates : 1351
  Train plates :  811  →   15747 colonies  (~60%)
  Val   plates :  270  →    4864 colonies  (~19%)
  Test  plates :  270  →    5602 colonies  (~21%)
  Leakage checks: PASSED
Split restored.
  TRAIN_PLATES : 811
  VAL_PLATES   : 270
  TEST_PLATES  : 270


In [ ]:
# ── Free GPU memory from Single Colony CNN ──────────────────
import gc
if 'model' in dir() and hasattr(model, 'parameters'):
    model.cpu()          # move weights off GPU
    del model            # remove reference
if 'single_cnn' in dir():
    single_cnn.cpu()
    del single_cnn
gc.collect()
torch.cuda.empty_cache()
print(f'GPU free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')
# ────────────────────────────────────────────────────────────

GPU free: 15.53 GB


In [ ]:
# ==================== MAIN — SIAMESE CNN ====================
# Uses TRAIN_PLATES / VAL_PLATES from the plate-aware split above
# so the Siamese sees the same plate division as the SingleColonyCNN.
torch.cuda.empty_cache()

METADATA_PATH    = os.path.join(WORK_DIR,   'metadata.json')
DATASET_PATH     = os.path.join(WORK_DIR,   'Dataset')
SIAMESE_SAVE_PATH   = os.path.join(OUTPUT_DIR, 'best_siamese_model.pth')
SIAMESE_CKPT_PATH   = os.path.join(OUTPUT_DIR, 'siamese_checkpoint_resume.pth')

SIAMESE_BATCH_SIZE  = 64
SIAMESE_NUM_EPOCHS  = 50
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

# Generate pairs restricted to train plates for training,
# and val plates for validation — consistent with the CNN split.
print('=== Generating TRAIN pairs ===')
tr_pairs, tr_labels = generate_pairs_from_json(
    METADATA_PATH, max_pairs_per_plate=50,
    restrict_to_plates=TRAIN_PLATES
)
print('\n=== Generating VAL pairs ===')
va_pairs, va_labels = generate_pairs_from_json(
    METADATA_PATH, max_pairs_per_plate=50,
    restrict_to_plates=VAL_PLATES
)

transform = transforms.Compose([
    AdaptivePadding(20),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])
tr_siam_ds = SiameseDataset(tr_pairs, tr_labels, DATASET_PATH, transform)
va_siam_ds = SiameseDataset(va_pairs, va_labels, DATASET_PATH, transform)
tr_siam_loader = DataLoader(tr_siam_ds, SIAMESE_BATCH_SIZE,
                             shuffle=True,  num_workers=2)
va_siam_loader = DataLoader(va_siam_ds, SIAMESE_BATCH_SIZE,
                             shuffle=False, num_workers=2)

siamese_model = SiameseCNN(embedding_dim=15)
if not os.path.exists(SIAMESE_CKPT_PATH):
    siamese_model.apply(init_weights)

siamese_history = train_siamese_network(
    siamese_model, tr_siam_loader, va_siam_loader,
    SIAMESE_NUM_EPOCHS, DEVICE,
    SIAMESE_SAVE_PATH, SIAMESE_CKPT_PATH
)

with open(os.path.join(OUTPUT_DIR, 'siamese_history.pkl'), 'wb') as f:
    pickle.dump(siamese_history, f)
print('Siamese training complete.')

=== Generating TRAIN pairs ===
Generating pairs (max 50 genuine per plate)...
  Restricted to 811 plates → 15747 colonies
  Genuine pairs  : 23084
  Impostor pairs : 23084
  Total          : 46168 pairs  (50/50 balance)

=== Generating VAL pairs ===
Generating pairs (max 50 genuine per plate)...
  Restricted to 270 plates → 4864 colonies
  Genuine pairs  : 7452
  Impostor pairs : 7452
  Total          : 14904 pairs  (50/50 balance)
  [DEBUG] checkpoint_path = '/content/drive/MyDrive/TFG/resultados_entrenamiento/siamese_checkpoint_resume.pth'
  [DEBUG] file exists     = True
Resuming from epoch 29
  [best so far: 0.0376]


Epoch 30/50: 100%|██████████| 722/722 [05:33<00:00,  2.16it/s]


Epoch 30: Train Loss=0.0451 Acc=89.1% | Val Loss=0.0388 Acc=85.3%
  [best so far: 0.0376]


Epoch 31/50: 100%|██████████| 722/722 [05:49<00:00,  2.06it/s]


Epoch 31: Train Loss=0.0451 Acc=89.0% | Val Loss=0.0448 Acc=82.4%
  [best so far: 0.0376]


...

Epoch 48: Train Loss=0.0438 Acc=89.6% | Val Loss=0.0392 Acc=86.4%
  [best so far: 0.0376]


Epoch 49/50: 100%|██████████| 722/722 [05:50<00:00,  2.06it/s]


Epoch 49: Train Loss=0.0440 Acc=89.6% | Val Loss=0.0387 Acc=85.9%
  [best so far: 0.0376]


Epoch 50/50: 100%|██████████| 722/722 [05:52<00:00,  2.05it/s]


Epoch 50: Train Loss=0.0440 Acc=89.5% | Val Loss=0.0375 Acc=88.0%
  → Best model saved (Val Loss: 0.0375)
Siamese training complete.


### Training Curves — Siamese CNN

In [ ]:
# Load and plot Siamese history (no retraining needed)
siam_history_path = os.path.join(OUTPUT_DIR, 'siamese_history.pkl')
if os.path.exists(siam_history_path):
    with open(siam_history_path, 'rb') as f:
        _sh = pickle.load(f)
    plot_training_curves(_sh, title='Siamese CNN — Training Curves',
                         save_dir=OUTPUT_DIR)
else:
    print('Siamese history not found — run training first.')

NameError: name 'os' is not defined

In [ ]:
ckpt = torch.load(SIAMESE_CKPT_PATH, map_location='cpu')
losses = ckpt['history']['train_losses']
print([round(l, 6) for l in losses])

[4.390032, 0.108547, 0.095785, 0.088256, 0.081357, 0.077615, 0.076419, 0.073556, 0.07102, 0.067494, 0.063736, 0.063467, 0.062009, 0.062749, 0.062349, 0.061449, 0.06208, 0.060666, 0.059647, 0.059862, 0.062117, 0.060238, 0.059462, 0.060303, 0.06007, 0.059377, 0.060287, 0.060551, 0.060085, 0.060245, 0.060872, 0.059647, 0.059609, 0.059112, 0.058934, 0.058531, 0.058402, 0.058454, 0.058475, 0.058381, 0.058725, 0.058478, 0.057853, 0.058525, 0.057272, 0.057788, 0.059425, 0.058609, 0.057869, 0.058064]


---
## Level 3 — Inference: Embedding + Clustering + Smoothing

In [ ]:
# ==================== LEVEL 3 — EMBEDDING + CLUSTERING + SMOOTHING ====================
# Step 1 – Embed : one S-CNN branch → 15-dim vector per good colony
# Step 2 – Cluster: Mean-Shift on embedding space → strain groups
# Step 3 – Smooth : average pIDv's within each cluster → consensus ID

TRANSFORM_INFERENCE = transforms.Compose([
    AdaptivePadding(margin=20),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])


def get_pIDv_batch(image_paths, single_cnn, device, batch_size=32):
    """Run SingleColonyCNN on a list of paths → [M, 32] softmax outputs."""
    single_cnn.eval()
    all_pIDv = []
    for i in range(0, len(image_paths), batch_size):
        batch = image_paths[i:i + batch_size]
        imgs  = torch.stack([
            TRANSFORM_INFERENCE(Image.open(p).convert('RGB')) for p in batch
        ]).to(device)
        with torch.no_grad():
            pIDv = torch.softmax(single_cnn(imgs), dim=1).cpu().numpy()
        all_pIDv.append(pIDv)
    return np.vstack(all_pIDv)  # [M, 32]


def get_embeddings_batch(image_paths, siamese_cnn, device, batch_size=32):
    """Run one S-CNN branch → [M, 15] embedding matrix."""
    siamese_cnn.eval()
    all_emb = []
    for i in range(0, len(image_paths), batch_size):
        batch = image_paths[i:i + batch_size]
        imgs  = torch.stack([
            TRANSFORM_INFERENCE(Image.open(p).convert('RGB')) for p in batch
        ]).to(device)
        with torch.no_grad():
            emb = siamese_cnn.forward_one(imgs).cpu().numpy()
        all_emb.append(emb)
    return np.vstack(all_emb)  # [M, 15]


def cluster_with_meanshift(embeddings, bandwidth=None, quantile=0.3):
    """Mean-Shift clustering on colony embeddings → cluster label per colony."""
    if len(embeddings) == 1:
        return np.array([0])
    if bandwidth is None:
        bandwidth = estimate_bandwidth(
            embeddings, quantile=quantile,
            n_samples=min(len(embeddings), 500)
        )
        bandwidth = max(bandwidth, 1e-3)
    ms = MeanShift(bandwidth=bandwidth, bin_seeding=True, cluster_all=True)
    ms.fit(embeddings)
    n = len(np.unique(ms.labels_))
    print(f'    Mean-Shift: {len(embeddings)} colonies → {n} cluster(s) '
          f'(bw={bandwidth:.4f})')
    return ms.labels_


def smooth_pIDv(pIDv_matrix, cluster_labels):
    """Average pIDv within each cluster → consensus sIDv per colony."""
    sIDv = np.zeros_like(pIDv_matrix)
    for cid in np.unique(cluster_labels):
        mask       = (cluster_labels == cid)
        mean_pIDv  = pIDv_matrix[mask].mean(axis=0)
        mean_pIDv /= mean_pIDv.sum() + 1e-9
        sIDv[mask] = mean_pIDv
        top1       = np.argmax(mean_pIDv)
        print(f'    Cluster {cid}: {mask.sum()} colonies | '
              f'top-1 idx={top1} (conf={mean_pIDv[top1]:.3f})')
    return sIDv


def run_level3(image_paths, pIDv_matrix, siamese_cnn, device,
               bandwidth=None, quantile=0.3):
    """Full Level 3 pipeline for one plate."""
    print(f'  Level 3 | {len(image_paths)} good colonies')
    print('  [1/3] Generating 15-dim embeddings...')
    embeddings = get_embeddings_batch(image_paths, siamese_cnn, device)

    print('  [2/3] Mean-Shift clustering...')
    cluster_labels = cluster_with_meanshift(embeddings, bandwidth, quantile)

    print('  [3/3] Smoothing pIDv within clusters...')
    sIDv_matrix = smooth_pIDv(pIDv_matrix, cluster_labels)

    n_clusters = len(np.unique(cluster_labels))
    changed    = int((np.argmax(pIDv_matrix, 1) != np.argmax(sIDv_matrix, 1)).sum())
    print(f'  Done: {n_clusters} group(s), {changed}/{len(image_paths)} IDs updated')
    return sIDv_matrix, cluster_labels, embeddings

---
## Level 4 — Culture Significance Assessment

In [ ]:
# ==================== LEVEL 4 — CULTURE SIGNIFICANCE ====================
# Rules from Fig.5a:
#   1. total colonies < 10         → No significant growth
#   2. detected species ≤ 2        → Significant
#   3. detected species ≥ 3:
#       a. one pathogen > 50 CFU AND all NGUF < 50 CFU → Significant
#       b. otherwise → Contaminated

SPECIES_NAMES = [
    'Staphylococcus aureus',       'Staphylococcus epidermidis',
    'Staphylococcus simulans',     'Staphylococcus lugdunensis',
    'Staphylococcus saprophyticus','Streptococcus mitis',
    'Streptococcus oralis',        'Streptococcus agalactiae',
    'Streptococcus pyogenes',      'Streptococcus dysgalactiae (Gr. C/G)',
    'Enterococcus faecalis',       'Enterococcus faecium',
    'Aerococcus urinae',           'Aerococcus sanguicola',
    'Lactobacillus species',       'Pseudomonas aeruginosa',
    'Escherichia coli',            'Citrobacter koseri',
    'Citrobacter freundii',        'Klebsiella pneumoniae',
    'Klebsiella oxytoca',          'Enterobacter aerogenes',
    'Enterobacter cloacae',        'Serratia marcescens',
    'Providencia stuartii',        'Proteus mirabilis',
    'Morganella morganii',         'Candida albicans',
    'Candida parapsilosis',        'Candida tropicalis',
    'Candida krusei',              'Candida glabrata',
]

NGUF_SPECIES = {
    'Lactobacillus species',    'Staphylococcus epidermidis',
    'Staphylococcus simulans',  'Staphylococcus lugdunensis',
    'Streptococcus mitis',      'Streptococcus oralis',
    'Aerococcus urinae',        'Aerococcus sanguicola',
}


def estimate_cfu(sIDv_matrix, species_names=SPECIES_NAMES):
    """Count good colonies per species using Top-1 of each sIDv."""
    counts = defaultdict(int)
    for sIDv in sIDv_matrix:
        counts[species_names[np.argmax(sIDv)]] += 1
    return dict(counts)


def assess_plate_significance(total_colonies, cfu_per_species,
                               nguf_species=NGUF_SPECIES,
                               min_threshold=10, prevalence_threshold=50):
    """Apply CML rules from Fig.5a to determine plate significance."""
    sorted_species = sorted(cfu_per_species.items(),
                            key=lambda x: x[1], reverse=True)
    n_species = len(sorted_species)

    if total_colonies < min_threshold:
        return {'significance': 'No significant growth', 'species': sorted_species,
                'reason': f'Total colonies ({total_colonies}) < {min_threshold}'}

    if n_species <= 2:
        return {'significance': 'Significant', 'species': sorted_species,
                'reason': f'{n_species} species detected'}

    for sp, cfu in sorted_species:
        if sp not in nguf_species and cfu >= prevalence_threshold:
            nguf_all_low = all(
                c < prevalence_threshold
                for s, c in sorted_species if s in nguf_species
            )
            if nguf_all_low:
                return {'significance': 'Significant', 'species': sorted_species,
                        'reason': f'{sp} dominant ({cfu} CFU) over NGUF flora'}

    return {'significance': 'Contaminated', 'species': sorted_species,
            'reason': f'{n_species} species, no dominant pathogen'}


def suggest_pick(image_paths, sIDv_matrix, significance_result,
                 species_names=SPECIES_NAMES):
    """For each significant species, return the colony path with highest confidence."""
    if significance_result['significance'] == 'No significant growth':
        return {}
    picks = {}
    for target_sp, _ in significance_result['species']:
        t_idx = (species_names.index(target_sp)
                 if target_sp in species_names else -1)
        if t_idx < 0:
            continue
        candidates = [
            (i, sIDv_matrix[i][t_idx])
            for i in range(len(sIDv_matrix))
            if np.argmax(sIDv_matrix[i]) == t_idx
        ]
        if candidates:
            best = max(candidates, key=lambda x: x[1])
            picks[target_sp] = image_paths[best[0]]
    return picks


def run_level4(image_paths, sIDv_matrix, cluster_labels,
               total_colonies_on_plate, species_names=SPECIES_NAMES):
    """Full Level 4 pipeline for one plate."""
    print(f'  Level 4 | total colonies on plate: {total_colonies_on_plate}')
    cfu          = estimate_cfu(sIDv_matrix, species_names)
    significance = assess_plate_significance(total_colonies_on_plate, cfu)
    picks        = suggest_pick(image_paths, sIDv_matrix, significance, species_names)
    print(f'  ► Significance : {significance["significance"]}')
    print(f'  ► Reason       : {significance["reason"]}')
    print(f'  ► Species CFU  : {dict(list(cfu.items())[:5])}'
          f'{"..." if len(cfu) > 5 else ""}')
    if picks:
        print(f'  ► Pick colonies: {list(picks.keys())}')
    return {**significance, 'cfu': cfu, 'picks': picks}

---
## Full Pipeline: Levels 2 → 3 → 4

In [ ]:
# ==================== FULL PIPELINE: LEVELS 2→3→4 ====================
def run_full_pipeline(image_paths, total_colonies_on_plate,
                      single_cnn, siamese_cnn, device,
                      species_names=SPECIES_NAMES,
                      bandwidth=None, quantile=0.3):
    """
    End-to-end inference for a single plate.
    Called AFTER training is complete — no retraining ever needed.

    Args:
        image_paths             : list of good-colony image paths (Level 1 output)
        total_colonies_on_plate : total count from Level 0 segmentation map
        single_cnn              : trained SingleColonyCNN (loaded from best model)
        siamese_cnn             : trained SiameseCNN (loaded from best model)
        device                  : torch device
        species_names           : ordered list matching model's idx_to_species
        bandwidth               : Mean-Shift bandwidth (None = auto-estimate)
        quantile                : quantile for bandwidth estimation

    Returns:
        dict with pIDv, sIDv, clusters, embeddings + Level 4 results
    """
    print('=' * 60)
    print(f'DeepColony Pipeline  |  {len(image_paths)} good colonies')
    print('=' * 60)

    print('\nLevel 2: Single colony identification...')
    pIDv_matrix = get_pIDv_batch(image_paths, single_cnn, device)
    top1_before = [species_names[i] for i in np.argmax(pIDv_matrix, axis=1)]
    print(f'  Species before smoothing: {set(top1_before)}')

    print('\nLevel 3: Context-based identification...')
    sIDv_matrix, clusters, embeddings = run_level3(
        image_paths, pIDv_matrix, siamese_cnn, device, bandwidth, quantile
    )

    print('\nLevel 4: Culture significance assessment...')
    result = run_level4(
        image_paths, sIDv_matrix, clusters,
        total_colonies_on_plate, species_names
    )

    return {
        'pIDv'      : pIDv_matrix,
        'sIDv'      : sIDv_matrix,
        'clusters'  : clusters,
        'embeddings': embeddings,
        **result
    }


# ==================== LOAD TRAINED MODELS FOR INFERENCE ====================
# Run this cell to load both models from Drive after any disconnection.
# No retraining is ever needed once best_single_colony_model.pth and
# best_siamese_model.pth exist in OUTPUT_DIR.

def load_trained_models(single_model_path, siamese_model_path, device):
    """Load both trained models from disk. Returns (single_cnn, siamese_cnn, species_names)."""
    # Single Colony CNN
    ckpt_single = torch.load(single_model_path, map_location=device)
    single_cnn  = SingleColonyCNN().to(device)
    single_cnn.load_state_dict(ckpt_single['model_state_dict'])
    single_cnn.eval()
    # Recover species order as it was at training time
    idx_to_species  = ckpt_single['idx_to_species']
    species_names   = [idx_to_species[i] for i in range(len(idx_to_species))]
    print(f'SingleColonyCNN loaded  (epoch {ckpt_single["epoch"]}, '
          f'val_acc={ckpt_single.get("val_acc", "?"):.2f}%)')

    # Siamese CNN
    siamese_cnn = SiameseCNN(embedding_dim=15).to(device)
    siamese_cnn.load_state_dict(torch.load(siamese_model_path, map_location=device))
    siamese_cnn.eval()
    print('SiameseCNN loaded')

    return single_cnn, siamese_cnn, species_names


# Uncomment once both model files exist:
# DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# single_cnn, siamese_cnn, SPECIES_NAMES_ORDERED = load_trained_models(
#     MODEL_SAVE_PATH, SIAMESE_SAVE_PATH, DEVICE
# )
#
# plate_colony_paths    = ['/content/data_lab/Dataset/colony_001.jpg', ...]
# total_colonies_level0 = 38
#
# results = run_full_pipeline(
#     image_paths             = plate_colony_paths,
#     total_colonies_on_plate = total_colonies_level0,
#     single_cnn              = single_cnn,
#     siamese_cnn             = siamese_cnn,
#     device                  = DEVICE,
#     species_names           = SPECIES_NAMES_ORDERED,
# )
# print(f'Result: {results["significance"]}  |  {len(results["picks"])} pick(s)')

---
## (Optional) Bandwidth Optimisation on Virtual Plates

In [ ]:
# ==================== BANDWIDTH OPTIMISATION ====================
# Run ONCE after Siamese training to find the optimal Mean-Shift bandwidth.
# Paper: 1000 virtual mixed-flora plates (up to 3 strains each),
# selecting bandwidth that maximises median v-measure.

def create_virtual_plates(metadata_path, n_plates=1000, max_strains=3, seed=42):
    random.seed(seed)
    with open(metadata_path) as f:
        data = json.load(f)
    samples  = list(data['patch_list'].values())
    by_strain = defaultdict(list)
    for s in samples:
        by_strain[s['species']].append(s['filename'])
    strains = [s for s, imgs in by_strain.items() if len(imgs) >= 3]

    plates = []
    for _ in range(n_plates):
        n      = random.randint(1, max_strains)
        chosen = random.sample(strains, min(n, len(strains)))
        cols   = []
        for st in chosen:
            k    = random.randint(3, min(15, len(by_strain[st])))
            cols += [{'path': p, 'strain': st}
                     for p in random.sample(by_strain[st], k)]
        plates.append(cols)
    print(f'Created {len(plates)} virtual plates')
    return plates, by_strain


def optimise_bandwidth(virtual_plates, dataset_path, siamese_cnn, device,
                        candidates=None):
    if candidates is None:
        candidates = np.logspace(-2, 1, 15).tolist()
    best_bw, best_vm = candidates[0], -1.0
    for bw in candidates:
        vms = []
        for plate in virtual_plates[:200]:
            paths  = [os.path.join(dataset_path, c['path']) for c in plate]
            labels = [c['strain'] for c in plate]
            if len(paths) < 2:
                continue
            try:
                emb  = get_embeddings_batch(paths, siamese_cnn, device)
                pred = cluster_with_meanshift(emb, bandwidth=bw)
                vms.append(v_measure_score(labels, pred))
            except Exception:
                continue
        if vms:
            med = float(np.median(vms))
            print(f'  bw={bw:.4f}  median v-measure={med:.4f}')
            if med > best_vm:
                best_vm, best_bw = med, bw
    print(f'\n✓ Best bandwidth: {best_bw:.4f}  (v-measure={best_vm:.4f})')
    return best_bw


# Usage (uncomment after Siamese training):
# virtual_plates, _ = create_virtual_plates(METADATA_PATH)
# best_bw = optimise_bandwidth(virtual_plates, DATASET_PATH, siamese_cnn, DEVICE)
# # Then: run_full_pipeline(..., bandwidth=best_bw)